In [1]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# 1. Prepare Twitter Dataset
print("Loading online dataset...")
dataset_url = "https://raw.githubusercontent.com/satyamk28/Twitter-Sentiment-Analysis/master/twitter_training.csv"
df = pd.read_csv(dataset_url, names=['id', 'entity', 'sentiment', 'text'])

# Filter out non-binary classifications to keep it fast and accurate
df = df.dropna(subset=['text', 'sentiment'])
df = df[df['sentiment'].str.lower().isin(['positive', 'negative'])]
df['label'] = df['sentiment'].str.lower().map({'negative': 0, 'positive': 1})

# Pull a lightweight slice for rapid prototyping
train_df = df.sample(n=100, random_state=42)
eval_df = df.sample(n=50, random_state=24)

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

# 2. Tokenization with bert-base-uncased
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    text_list = [str(t) for t in examples["text"]]
    return tokenizer(text_list, truncation=True, padding="max_length", max_length=64)

print("Tokenizing data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# 3. Model setup
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

# 4. Training configuration
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# 5. Fine-Tune & Evaluate
print("\nTraining BERT...")
trainer.train()

print("\nEvaluating and calculating Confusion Matrix...")
predictions_output = trainer.predict(tokenized_eval)
preds = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

# Generate the confusion matrix array
cm = confusion_matrix(true_labels, preds)

print("\n--- RESULTS FOR ASSIGNMENT 1 ---")
print(f"Accuracy: {accuracy_score(true_labels, preds):.4f}")
print(f"F1-Score: {f1_score(true_labels, preds, average='weighted'):.4f}")
print("\nConfusion Matrix Output:")
print(f"True Negatives (TN): {cm[0][0]} | False Positives (FP): {cm[0][1]}")
print(f"False Negatives (FN): {cm[1][0]} | True Positives (TP): {cm[1][1]}")

Loading online dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Evaluating and calculating Confusion Matrix...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- RESULTS FOR ASSIGNMENT 1 ---
Accuracy: 0.6600
F1-Score: 0.6610

Confusion Matrix Output:
True Negatives (TN): 13 | False Positives (FP): 8
False Negatives (FN): 9 | True Positives (TP): 20
